# HP 모형 ver 3.0 — 시계열 구동 노트북 (HP_timeseries)

10분 간격 침수해석 결과(`H_(t)MIN.asc`)를 시간 순서대로 읽어 **공통 전처리 → 침수위험등급 → 회복력·복구우선순위**를 순차 수행하고, 시간단계별 결과 폴더와 구동 로그를 저장한다.

| 단계 | 모듈 | 기능 |
|---|---|---|
| 설정 | `hp_config.py` | 입력·출력 경로, 모형 파라미터 (**경로 변경은 여기서만**) |
| 1단계 공통 전처리 | `hp_common.py` | ASC 판독 → 건물·도로 침수심 추출 → 자산가치·취약도·피해액 산정 |
| 2단계 침수위험등급 | `hp_risk.py` | Hazard·Vulnerability·Exposure → 위험지수(Distance) → 4단계 위험등급 |
| 3단계 회복력·복구우선순위 | `hp_resilience.py` | 견고성·자원부존성·도로망 RA/SR/RI/TCC → Resilience → Restore_Priority |
| 러너 | `hp_runner.py` | 시간단계별 구동, 단계별 소요시간 기록 |

ver 2.0 (`HP_timeseries.ipynb` → `HP_건물_V3.ipynb` · `HP_도로.ipynb` 를 nbconvert 로 호출) 과 **연산식·상수·결과값은 동일**하며, 결과파일 이름과 열 순서도 ver 2.0 을 따른다.

### 시간단계별 산출 결과파일 (`OUTPUT\TimeSeries\H_(t)MIN\`)

| 파일 | 내용 | ver 2.0 대비 |
|---|---|---|
| `H_(t)MIN_results_building.csv` | 행정동별 피해액·재난복구지원금·예측이재민·견고성(건물)·자원부존성·위험등급 분포 | 동일 |
| `H_(t)MIN_results_road.csv` | 행정동별 도로 위험등급 분포·최단경로 수·TCC·RA·SR·RI | 동일 |
| `H_(t)MIN_results_building_risk.csv` | 건물별 침수심·피해액·위험인자·위험지수(Distance)·위험등급 | 동일 + `Grade` 열 추가(맨 뒤) |
| `H_(t)MIN_region_results_final.csv` | 행정동별 Robustness·Rapidness·Redundancy·Resourcefulness·**Resilience** | 동일 |
| `H_(t)MIN_building_risk_final.csv` | 건물별 Resilience·**Restore_Priority**·순위 (건물 기초자료 순서) | 동일 + `Grade` 열 추가(맨 뒤) |
| `H_(t)MIN_Priority.csv` | 위 파일을 복구우선순위 순위 오름차순으로 정렬 | 동일 + `Grade` 열 추가(맨 뒤) |
| `H_(t)MIN_results_road_risk.csv` | 도로 노드별 침수심·손상가중치·위험지수·위험등급 | **신규** |
| `H_(t)MIN_region_risk_grade.csv` | 행정동별 건물·도로 위험등급 분포 | **신규** |

`Grade` 는 위험등급 번호(1 매우낮음 · 2 낮음 · 3 위험 · 4 매우위험)로, 문자열 열 `Risk_Level` 과 같은 내용을 숫자로 표기한 것이다.


In [ ]:
import os, sys, time
import pandas as pd

# 모듈 폴더 경로 (본 노트북과 같은 폴더에 있으면 수정 불필요)
MODULE_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)

import hp_config as cfg
import hp_runner as runner

# ---- 구동 조건 --------------------------------------------------------------
ASC_DIR = cfg.ASC_INPUT_DIR               # 입력 ASC 폴더 (hp_config.py 에서 변경)
N_STEPS = None                            # None = 폴더 내 ASC 전체, 숫자 = 앞에서부터 N개만
OUT_DIR = cfg.ensure_output_dir("TimeSeries")   # 결과 저장 폴더 (시간단계별 하위폴더 생성)
N_JOBS  = -1                              # 도로망 병렬처리 코어 수 (-1 = 전체 논리코어)

asc_files = runner.list_asc_files(asc_dir=ASC_DIR, n_steps=N_STEPS)
print(f"입력 ASC {len(asc_files)}개  ({asc_files[0]} ~ {asc_files[-1]})")
print("입력 폴더 :", ASC_DIR)
print("결과 저장 :", OUT_DIR)


In [ ]:
# ---- 회차 준비 (1회) ---------------------------------------------------------
#  · 기초자료·학습모델 적재
#  · 도로망 기준값(침수 전 최단경로 등) 산정 — 시간단계가 바뀌어도 값이 같으므로 1회만 수행
ctx = runner.prepare_context(n_jobs=N_JOBS)
print(f"기초자료·학습모델 적재 : {ctx['load_seconds']:.3f} 초")
print(f"도로망 기준값 산정     : {ctx['road_baseline_seconds']:.3f} 초")
print(f"회차 준비 합계         : {ctx['setup_seconds']:.3f} 초  (시간단계별 구동시간에 미포함)")


In [ ]:
# ---- 시계열 순차 구동 --------------------------------------------------------
logs = []
for k, name in enumerate(asc_files, 1):
    timing, _ = runner.run_step(ctx, name, out_dir=OUT_DIR, save=True, n_jobs=N_JOBS, asc_dir=ASC_DIR)
    timing["시간단계_k"] = k
    logs.append(timing)
    flag = "OK " if (timing["위험등급_정상종료"] and timing["회복력_정상종료"]) else "FAIL"
    print(f"[{flag}] k={k:>3}/{len(asc_files)}  {timing['입력자료']:<12} "
          f"위험등급={timing['T_RISK_위험등급_구동시간']:7.2f}s  "
          f"회복력={timing['T_RES_회복력_구동시간']:7.2f}s  "
          f"합계={timing['T_ALL_통합']:7.2f}s")

df_log = pd.DataFrame(logs)
df_log["회차준비_기초자료적재_s"] = ctx['load_seconds']
df_log["회차준비_도로망기준값_s"] = ctx['road_baseline_seconds']
log_path = os.path.join(OUT_DIR, "HP_시계열구동_로그.csv")
df_log.to_csv(log_path, index=False, encoding='utf-8-sig')
print("\n[저장]", log_path)
df_log[["시간단계_k", "입력자료", "T_C_공통전처리", "T_G_위험등급", "T_R_회복력",
        "T_RISK_위험등급_구동시간", "T_RES_회복력_구동시간", "T_ALL_통합"]].head(10)


In [ ]:
# ---- 결과 확인 : 마지막 시간단계 결과 폴더의 파일 목록과 행정동별 Resilience ----------
last = asc_files[-1].replace('.asc', '')
last_dir = os.path.join(OUT_DIR, last)
print(f"[{last}] 결과 파일 :")
for f in sorted(os.listdir(last_dir)):
    print("   ", f, f"({os.path.getsize(os.path.join(last_dir, f))/1024:,.0f} KB)")

region = pd.read_csv(os.path.join(last_dir, f"{last}_region_results_final.csv"), encoding='cp949')
region[["ADM_CD", "ADM_NM", "Robustness", "Rapidness", "Redundancy", "Resourcefulness", "Resilience"]]
